<a href="https://colab.research.google.com/github/falvarezaguiar/PC6_RAG_scratch_Constitucion/blob/main/PC6_rag_scratch_Constitucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crear y ejecutar un pipeline RAG local desde cero

El objetivo de este notebook es construir un pipeline RAG (Retrieval Augmented Generation) desde cero.

Específicamente, queremos ser capaces de abrir un archivo PDF, hacer preguntas (queries) sobre él y obtener respuestas generadas por un Large Language Model (LLM).

Existen frameworks que replican este tipo de flujo de trabajo, incluyendo [LlamaIndex](https://www.llamaindex.ai/) y [LangChain](https://www.langchain.com/), sin embargo, el objetivo de construirlo desde cero es poder inspeccionar y personalizar todas las partes.

## ¿Qué es RAG?

RAG significa Retrieval Augmented Generation (Generación Aumentada por Recuperación).

Fue introducido en el paper [*Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*](https://arxiv.org/abs/2005.11401).

Cada paso puede desglosarse aproximadamente en:

* **Retrieval (Recuperación)** - Buscar información relevante de una fuente dada una consulta. Por ejemplo, obtener pasajes relevantes de texto de Wikipedia desde una base de datos dada una pregunta.
* **Augmented (Aumentado)** - Usar la información relevante recuperada para modificar una entrada a un modelo generativo (ej. un LLM).
* **Generation (Generación)** - Generar una salida dada una entrada. Por ejemplo, en el caso de un LLM, generar un pasaje de texto dado un prompt de entrada.

## ¿Por qué RAG?

El objetivo principal de RAG es mejorar las salidas de generación de los LLMs.

Dos mejoras principales pueden verse como:
1. **Prevenir alucinaciones** - Los LLMs son increíbles pero son propensos a posibles alucinaciones, es decir, generar algo que *parece* correcto pero no lo es. Los pipelines RAG pueden ayudar a los LLMs a generar salidas más factuales proporcionándoles entradas factuales (recuperadas). E incluso si la respuesta generada de un pipeline RAG no parece correcta, debido a la recuperación, también tienes acceso a las fuentes de donde provino.
2. **Trabajar con datos personalizados** - Muchos LLMs base están entrenados con datos de texto a escala de internet. Esto significa que tienen una gran capacidad para modelar el lenguaje, sin embargo, a menudo carecen de conocimiento específico. Los sistemas RAG pueden proporcionar a los LLMs datos específicos del dominio como información médica o documentación empresarial y así personalizar sus salidas para adaptarse a casos de uso específicos.


RAG también puede ser una solución mucho más rápida de implementar que hacer fine-tuning de un LLM con datos específicos.



## ¿Para qué tipo de problemas se puede usar RAG?

RAG puede ayudar en cualquier lugar donde exista un conjunto específico de información que un LLM pueda no tener en sus datos de entrenamiento (ej. cualquier cosa que no sea públicamente accesible en internet).

Por ejemplo, podrías usar RAG para:
* **Chat de Q&A de soporte al cliente** - Al tratar tu documentación de soporte al cliente existente como un recurso, cuando un cliente haga una pregunta, podrías tener un sistema que recupere fragmentos de documentación relevantes y luego hacer que un LLM convierta esos fragmentos en una respuesta. Piensa en esto como un "chatbot para tu documentación".
* **Análisis de cadenas de email** - Supongamos que eres una compañía de seguros con largos hilos de emails entre clientes y agentes de seguros. En lugar de buscar en cada email individual, podrías recuperar pasajes relevantes y hacer que un LLM cree salidas estructuradas de reclamaciones de seguros.
* **Chat de documentación interna de empresa** - Si has trabajado en una gran empresa, sabes lo difícil que puede ser obtener una respuesta a veces. ¿Por qué no dejar que un sistema RAG indexe la información de tu empresa y hacer que un LLM responda preguntas que puedas tener? El beneficio de RAG es que tendrás referencias a recursos para aprender más si la respuesta del LLM no es suficiente.
* **Q&A de libros de texto** - Supongamos que estás estudiando para tus exámenes y constantemente hojeando un gran libro de texto buscando respuestas a tus preguntas. RAG puede ayudar a proporcionar respuestas así como referencias para aprender más.

Todos estos tienen el tema común de recuperar recursos relevantes y luego presentarlos de una manera comprensible usando un LLM.




 ## Qué es lo que vamos a construir

Vamos a construir un pipeline RAG que nos permita chatear con un documento PDF, específicamente la **Constitución Española** (~37 páginas en español).

Escribiremos el código para:
1. Abrir un documento PDF (en este caso, la Constitución Española).
2. Formatear el texto del PDF listo para un modelo de embedding (división/chunking de texto).
3. Embeber todos los chunks de texto y convertirlos en representaciones numéricas.
4. Construir un sistema de recuperación que use búsqueda vectorial para encontrar chunks relevantes.
5. Crear un prompt que incorpore los fragmentos de texto recuperados.
6. Generar una respuesta en español a consultas sobre la Constitución Española.

## 1. Procesamiento de Documentos/Texto y Creación de Embeddings

Ingredientes:
* Documento PDF de elección.
* Modelo de embedding de elección.

Pasos:
1. Importar documento PDF.
2. Procesar texto para embedding (ej. dividir en chunks de sentencias).
3. Embeber chunks de texto con modelo de embedding.
4. Guardar embeddings en archivo para uso posterior

In [ ]:
# from google.colab import drive
import os
# drive.mount('/content/drive')

# Montar Google Drive - Ya no es necesario, descargaremos el PDF directamente
# if not os.path.exists('/content/drive'):
#     drive.mount('/content/drive')
#     print(" Google Drive montado")

Aquí vamos a cargar la Constitución Española que previamente me he descargado del BOE y puesto en mi drive / data

In [ ]:
# from google.colab import drive # Ya no es necesario
# import shutil # Ya no es necesario
import os
import requests
from tqdm.auto import tqdm # Usaremos tqdm para mostrar el progreso de la descarga

# Ruta donde guardaremos el PDF descargado localmente en Colab
pdf_path = "BOE-151_Constitucion_Espanola.pdf"
pdf_url = "https://www.boe.es/biblioteca_juridica/codigos/abrir_pdf.php?fich=151_Constitucion_Espanola.pdf"

# Descargar el PDF si no existe localmente
if not os.path.exists(pdf_path):
    print(f" Descargando Constitución Española desde {pdf_url}...")
    try:
        response = requests.get(pdf_url, stream=True)
        response.raise_for_status() # Lanzar un error para respuestas de estado incorrectas (4xx o 5xx)

        # Obtener el tamaño total del archivo para la barra de progreso
        total_size_in_bytes = int(response.headers.get('content-length', 0))
        block_size = 1024 # 1 Kibibyte

        with tqdm(total=total_size_in_bytes, unit='iB', unit_scale=True, desc=pdf_path.split('/')[-1]) as progress_bar:
            with open(pdf_path, 'wb') as file:
                for data in response.iter_content(block_size):
                    progress_bar.update(len(data))
                    file.write(data)

        if total_size_in_bytes != 0 and progress_bar.n != total_size_in_bytes:
            print("ERROR, la descarga no se completó correctamente")
        else:
            file_size = os.path.getsize(pdf_path) / 1024 / 1024
            print(f" PDF descargado exitosamente: {pdf_path} ({file_size:.2f} MB)")

    except requests.exceptions.RequestException as e:
        print(f" Error al descargar el PDF: {e}")
        print(" Por favor, verifica la URL o tu conexión a internet.")

else:
    print(f" PDF ya existe en Colab: {pdf_path}")

 PDF ya existe en Colab: BOE-151_Constitucion_Espanola.pdf


Aquí vamos a descargar la Constitución Española directamente desde la fuente original del BOE.

In [ ]:
!pip install PyMuPDF
!pip install tqdm

Aquí vamos extrayendo el texto de la Constitución pero limpiando un poco, replazo de espacios y saltos de lineas. Al final sacamos por pantalla la primera frase de la primera página y la primera frase de la segunda frase. Ajustamos también a 8 porque es donde realmente empieza el texto.

In [ ]:
import fitz #for opening document
from tqdm import tqdm

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text = text.replace("\n", " ").strip()
    return cleaned_text

def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """Opens a PDF file, reads its text content page by page, and collects statistics."""
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):
        text = page.get_text()
        text = text_formatter(text=text)
        pages_and_texts.append({"page_number": page_number - 8, # ajustamos saltandonos la introducción y la tabla de índice
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count_raw": len(text.split(", ")),
                                "page_token_count": len(text) / 4, #1 token has approx 4 characters
                                "text": text})
    return pages_and_texts
pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

191it [00:00, 304.12it/s]


[{'page_number': -8,
  'page_char_count': 114,
  'page_word_count': 16,
  'page_sentence_count_raw': 1,
  'page_token_count': 28.5,
  'text': 'Códigos electrónicos Constitución Española Edición actualizada a 8 de noviembre de 2024 BOLETÍN OFICIAL DEL ESTADO'},
 {'page_number': -7,
  'page_char_count': 721,
  'page_word_count': 94,
  'page_sentence_count_raw': 3,
  'page_token_count': 180.25,
  'text': 'La última versión de este Código en PDF y ePUB está disponible para su descarga gratuita en:  www.boe.es/biblioteca_juridica/ Alertas de actualización en Mi BOE: www.boe.es/mi_boe/ Para adquirir el Código en formato papel: tienda.boe.es Esta obra está sujeta a licencia Creative Commons de Reconocimiento-NoComercial-SinObraDerivada  4.0 Internacional, (CC BY-NC-ND 4.0). @ Agencia Estatal Boletín Oficial del Estado NIPO (PDF): 007-16-026-6 NIPO (ePUB): 007-16-027-1 NIPO (Papel): 007-16-028-7 ISBN: 978-84-340-2276-8 Depósito Legal: M-7183-2016 Catálogo de Publicaciones de la Administración 

In [ ]:
import random

random.sample(pages_and_texts, k=2)

[{'page_number': -5,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''},
 {'page_number': 85,
  'page_char_count': 3175,
  'page_word_count': 538,
  'page_sentence_count_raw': 20,
  'page_token_count': 793.75,
  'text': "amb el que estableixin els Estatuts, que asseguraran, en qualsevol cas, la representació  proporcional adequada. 6. El Senat és elegit per quatre anys. El mandat dels Senadors acaba quatre anys  després de l'elecció o el dia de la dissolució de la Cambra. Article 70.   1. La llei electoral determinarà les causes d'inelegibilitat i incompatibilitat dels Diputats i  dels Senadors, les quals comprendran, en qualsevol cas: a) Els components del Tribunal Constitucional. b) Els alts càrrecs de l'Administració de l'Estat que determini la llei, amb l'excepció dels  membres del Govern. c) El Defensor del Poble. d) Els Magistrats, els Jutges i els Fiscals en actiu. e) Els militars professionals i els membres d

Aquí descubrimos que la constitución está escrita en 5 lenguas castellano, catalán, gallego, euskera, valenciano. Por ello vamos a realizar un corte para quedarnos solo con el español.

In [ ]:
# Después de procesar el PDF
pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)

# AJUSTA ESTOS NÚMEROS según tu PDF
CASTELLANO_START = 0      # Primera página en castellano
CASTELLANO_END = 36       # Última página en castellano (ajustar según tu PDF)

# Filtrar solo páginas en castellano
pages_and_texts = [
    page for page in pages_and_texts
    if CASTELLANO_START <= page['page_number'] <= CASTELLANO_END
    and len(page['text']) > 100  # Mínimo 100 caracteres
]

print(f" Total páginas procesadas: {len(pages_and_texts)}")
print(f"🇪🇸 Páginas en castellano: {len(pages_and_texts)}")
print(f" Rango: páginas {CASTELLANO_START} a {CASTELLANO_END}")

# Ver una muestra
import random
sample = random.choice(pages_and_texts)
print(f" Ejemplo - Página {sample['page_number']}:")
print(sample['text'][:500])



191it [00:00, 440.80it/s]

 Total páginas procesadas: 37
🇪🇸 Páginas en castellano: 37
 Rango: páginas 0 a 36
 Ejemplo - Página 2:
TÍTULO I De los derechos y deberes fundamentales Artículo 10.   1. La dignidad de la persona, los derechos inviolables que le son inherentes, el libre  desarrollo de la personalidad, el respeto a la ley y a los derechos de los demás son  fundamento del orden político y de la paz social. 2. Las normas relativas a los derechos fundamentales y a las libertades que la  Constitución reconoce se interpretarán de conformidad con la Declaración Universal de  Derechos Humanos y los tratados y acuerdos in


### Obtener algunas estadísticas del texto

Realicemos un análisis exploratorio de datos (EDA) aproximado para obtener una idea del tamaño de los textos (ej. conteos de caracteres, conteos de palabras, etc.) con los que estamos trabajando.




In [ ]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,0,1688,282,17,422.00,§ 1 Constitución Española Cortes Generales «BO...
1,1,3050,515,18,762.50,Artículo 2. La Constitución se fundamenta en...
2,2,2588,444,20,647.00,TÍTULO I De los derechos y deberes fundamental...
3,3,3201,551,27,800.25,"la pena de muerte, salvo lo que puedan dispone..."
4,4,3372,570,28,843.00,2. El ejercicio de estos derechos no puede res...


In [ ]:
df.tail()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
32,32,3347,546,19,836.75,2. Las Comunidades Autónomas no podrán en ning...
33,33,3055,529,33,763.75,c) De los conflictos de competencia entre el E...
34,34,2663,443,23,665.75,3. Aprobada la reforma por las Cortes Generale...
35,35,3203,551,33,800.75,"prevé en el apartado 2 del artículo 148, cuand..."
36,36,2902,532,27,725.50,"Durante este período, el actual Presidente del..."


In [ ]:
df.shape

(37, 6)

In [ ]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,37.00,37.00,37.00,37.00,37.00
mean,18.00,3144.30,526.35,25.65,786.07
std,10.82,451.34,74.02,7.54,112.84
min,0.00,1688.00,282.00,10.00,422.00
25%,9.00,2917.00,491.00,20.00,729.25
50%,18.00,3118.00,530.00,25.00,779.50
75%,27.00,3372.00,570.00,31.00,843.00
max,36.00,4182.00,696.00,40.00,1045.50


### Procesamiento adicional de texto (división de páginas en sentencias)
Seguiremos el flujo de trabajo de:

`Ingerir texto -> dividirlo en grupos/chunks -> embeber los grupos/chunks -> usar los embeddings`

¿Por qué dividir en sentencias?

* Más fácil de manejar que páginas grandes de texto (especialmente si las páginas están densamente llenas de texto).
* Se puede ser específico y averiguar qué grupo de sentencias se usó para ayudar dentro de un pipeline RAG.


Usaremos spaCy para dividir nuestro texto en sentencias ya que es probablemente un poco más robusto que simplemente usar `text.split(". ")`.

In [ ]:
import spacy
from tqdm import tqdm

# Cargar el modelo de español de spaCy
try:
    nlp = spacy.load("es_core_news_sm")
except OSError:
    # Si el modelo no está instalado, descargarlo
    print("Descargando modelo de español de spaCy...")
    spacy.cli.download("es_core_news_sm")
    nlp = spacy.load("es_core_news_sm")

# Asegurarse de que el 'sentencizer' está en el pipeline
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")

for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    item["page_sentence_count_spacy"] = len(item["sentences"])

# Opcional: Mostrar un ejemplo para verificar
# import random
# print(random.sample(pages_and_texts, k=1))

100%|██████████| 37/37 [00:04<00:00,  7.79it/s]


In [ ]:
random.sample(pages_and_texts, k=1)

[{'page_number': 22,
  'page_char_count': 3347,
  'page_word_count': 575,
  'page_sentence_count_raw': 40,
  'page_token_count': 836.75,
  'text': 'Artículo 121.   Los daños causados por error judicial, así como los que sean consecuencia del  funcionamiento anormal de la Administración de Justicia, darán derecho a una  indemnización a cargo del Estado, conforme a la ley. Artículo 122.   1. La ley orgánica del poder judicial determinará la constitución, funcionamiento y  gobierno de los Juzgados y Tribunales, así como el estatuto jurídico de los Jueces y  Magistrados de carrera, que formarán un Cuerpo único, y del personal al servicio de la  Administración de Justicia. 2. El Consejo General del Poder Judicial es el órgano de gobierno del mismo. La ley  orgánica establecerá su estatuto y el régimen de incompatibilidades de sus miembros y sus  funciones, en particular en materia de nombramientos, ascensos, inspección y régimen  disciplinario. 3. El Consejo General del Poder Judicial estar

In [ ]:
df = pd.DataFrame(pages_and_texts)
df

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text,sentences,page_sentence_count_spacy
0,0,1688,282,17,422.00,§ 1 Constitución Española Cortes Generales «BO...,[§ 1 Constitución Española Cortes Generales «B...,12
1,1,3050,515,18,762.50,Artículo 2. La Constitución se fundamenta en...,"[Artículo 2. , La Constitución se fundamenta...",35
2,2,2588,444,20,647.00,TÍTULO I De los derechos y deberes fundamental...,[TÍTULO I De los derechos y deberes fundamenta...,26
3,3,3201,551,27,800.25,"la pena de muerte, salvo lo que puedan dispone...","[la pena de muerte, salvo lo que puedan dispon...",40
4,4,3372,570,28,843.00,2. El ejercicio de estos derechos no puede res...,"[2., El ejercicio de estos derechos no puede r...",37
5,5,3276,557,18,819.00,"de este Capítulo, a excepción de los que se ve...","[de este Capítulo, a excepción de los que se v...",39
6,6,2892,497,21,723.00,Sección 2.ª De los derechos y deberes de los c...,[Sección 2.ª De los derechos y deberes de los ...,38
7,7,2941,488,20,735.25,2. Se reconoce el derecho de los trabajadores ...,"[2., Se reconoce el derecho de los trabajadore...",33
8,8,3066,496,27,766.50,Artículo 44. 1. Los poderes públicos promove...,"[Artículo 44. 1., Los poderes públicos promo...",32
9,9,3268,548,38,817.00,2. Los poderes públicos promoverán la informac...,"[2., Los poderes públicos promoverán la inform...",25


In [ ]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,37.00,37.00,37.00,37.00,37.00,37.00
mean,18.00,3144.30,526.35,25.65,786.07,31.00
std,10.82,451.34,74.02,7.54,112.84,6.55
min,0.00,1688.00,282.00,10.00,422.00,12.00
25%,9.00,2917.00,491.00,20.00,729.25,27.00
50%,18.00,3118.00,530.00,25.00,779.50,32.00
75%,27.00,3372.00,570.00,31.00,843.00,36.00
max,36.00,4182.00,696.00,40.00,1045.50,40.00



mean: 3,144 caracteres/páginastd: 451 (±14% variación)min: 1,688 → max: 4,182mediana: 3,118
Interpretación:
- Páginas consistentes: Poca variación (±451 caracteres)
- La mayoría de páginas tienen ~3,000 caracteres
- Página más corta: 1,688 chars (probablemente final de sección/capítulo)
- Página más larga: 4,182 chars (sección densa)

Importante para los embedding en general son de 512 - 1024 Tokens, debemos de "chunkear" porque las páginas son demasiado largas. Por ello dividimos entre 10 chunks de 10 sentences.

### Agrupando nuestras sentencias juntas (chunking)
¿Por qué hacemos esto?

1. Es más fácil manejar "chunks" del mismo tamaño.
2. No sobrecargamos el modelo de embeddings con la capacidad de los tokens (por ejemplo, si la capacidad del embedding es de 384 tokens, entonces podríamos perder información si intentamos hacer embeddings con una secuencia de 400+ tokens).
3. La ventana de contexto del LLM se podría sobrepasar (la cantidad de tokens que un LLM puede recibir) además podría tener limitaciones y requerimientos computacionales más potentes, y lo que queremos es usarlos todos en la medida de lo posible.

In [ ]:
chunk_size = 10
def split_list(input_list: list[str],
               slice_size: int=chunk_size) -> list[list[str]]:
    """
    Splits the input_list into sublists of size slice_size (or as close as possible).

    For example, a list of 17 sentences would be split into two lists of [[10], [7]]
    """
    return [input_list[i:i+slice_size] for i in range(0, len(input_list), slice_size)]

for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                         slice_size=chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

100%|██████████| 37/37 [00:00<00:00, 142244.96it/s]


In [ ]:
random.sample(pages_and_texts,k=1)

[{'page_number': 14,
  'page_char_count': 3480,
  'page_word_count': 580,
  'page_sentence_count_raw': 38,
  'page_token_count': 870.0,
  'text': '2. Las decisiones de las Cortes Generales previstas en los artículos 94, 1, 145, 2 y 158,  2, se adoptarán por mayoría de cada una de las Cámaras. En el primer caso, el  procedimiento se iniciará por el Congreso, y en los otros dos, por el Senado. En ambos  casos, si no hubiera acuerdo entre Senado y Congreso, se intentará obtener por una  Comisión Mixta compuesta de igual número de Diputados y Senadores. La Comisión  presentará un texto que será votado por ambas Cámaras. Si no se aprueba en la forma  establecida, decidirá el Congreso por mayoría absoluta. Artículo 75.   1. Las Cámaras funcionarán en Pleno y por Comisiones. 2. Las Cámaras podrán delegar en las Comisiones Legislativas Permanentes la  aprobación de proyectos o proposiciones de ley. El Pleno podrá, no obstante, recabar en  cualquier momento el debate y votación de cualquier pro

In [ ]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,37.00,37.00,37.00,37.00,37.00,37.00,37.00
mean,18.00,3144.30,526.35,25.65,786.07,31.00,3.54
std,10.82,451.34,74.02,7.54,112.84,6.55,0.61
min,0.00,1688.00,282.00,10.00,422.00,12.00,2.00
25%,9.00,2917.00,491.00,20.00,729.25,27.00,3.00
50%,18.00,3118.00,530.00,25.00,779.50,32.00,4.00
75%,27.00,3372.00,570.00,31.00,843.00,36.00,4.00
max,36.00,4182.00,696.00,40.00,1045.50,40.00,4.00


### Dividiendo cada chunk en su propio elemento


In [ ]:
import re

pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]

        # Join the sentences together into a paragraph-like structure, cleaning a chunk (so they are a single string, Todo en miniusculas, excepto lo que viene después de un punto, elimina dobles espacios vacios, dejando una frase más limpia si cabe)
        joined_sentence_chunk = "".join(sentence_chunk).replace("  "," ").strip()
        joined_sentence_chunk = re.sub(r'\.(A-Z)', r'. \1', joined_sentence_chunk) # convert ".A"to ". A"(only for capital letter)
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4

        pages_and_chunks.append(chunk_dict)

len(pages_and_chunks)

100%|██████████| 37/37 [00:00<00:00, 9991.58it/s]


131

In [ ]:
random.sample(pages_and_chunks, k=1)

[{'page_number': 35,
  'sentence_chunk': 'prevé en el apartado 2 del artículo 148, cuando así lo acordaren, por mayoría absoluta, sus órganos preautonómicos colegiados superiores, comunicándolo al Gobierno.El proyecto de Estatuto será elaborado de acuerdo con lo establecido en el artículo 151, número 2, a convocatoria del órgano colegiado preautonómico.Tercera.  La iniciativa del proceso autonómico por parte de las Corporaciones locales o de sus miembros, prevista en el apartado 2 del artículo 143, se entiende diferida, con todos sus efectos, hasta la celebración de las primeras elecciones locales una vez vigente la Constitución.Cuarta.  1.En el caso de Navarra, y a efectos de su incorporación al Consejo General Vasco o al régimen autonómico vasco que le sustituya, en lugar de lo que establece el artículo 143 de la Constitución, la iniciativa corresponde al Órgano Foral competente, el cual adoptará su decisión por mayoría de los miembros que lo componen.Para la validez de dicha iniciat

Volvemos a obtener las estadísticas generales del df para verificar que no se ha "roto nada"

In [ ]:
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,131.00,131.00,131.00,131.00
mean,17.40,871.15,132.45,217.79
std,10.31,353.18,55.74,88.30
min,0.00,54.00,9.00,13.50
25%,8.50,706.00,108.00,176.50
50%,17.00,880.00,134.00,220.00
75%,26.00,1019.50,155.00,254.88
max,36.00,2225.00,390.00,556.25


In [ ]:
 df.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count
0,0,§ 1 Constitución Española Cortes Generales «BO...,1571,246,392.75
1,0,La forma política del Estado español es la Mon...,94,14,23.50
2,1,Artículo 2. La Constitución se fundamenta en ...,695,108,173.75
3,1,La bandera de España está formada por tres fra...,756,118,189.00
4,1,Artículo 7. Los sindicatos de trabajadores y ...,828,125,207.00


Aquí estamos revisando los fragmentos demasiado cortos con menosde 30 tokens (nos salen solo 3)

In [ ]:
min_token_length = 30
# Filtrar el DataFrame por el número mínimo de tokens y iterar sobre las filas
for row in df[df["chunk_token_count"] <= min_token_length].iterrows():
    print(f'Chunk token count : {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"]}')

Chunk token count : 23.5 | Text: La forma política del Estado español es la Monarquía parlamentaria.CONSTITUCIÓN ESPAÑOLA – 1 –
Chunk token count : 13.5 | Text: CONSTITUCIÓN ESPAÑOLA § 1 Constitución Española – 24 –
Chunk token count : 13.5 | Text: CONSTITUCIÓN ESPAÑOLA § 1 Constitución Española – 32 –


In [ ]:
#filtering rows with token under 30
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_token_len[:2]

[{'page_number': 0,
  'sentence_chunk': '§ 1 Constitución Española Cortes Generales «BOE» núm. 311, de 29 de diciembre de 1978 Última modificación: 17 de febrero de 2024 Referencia: BOE-A-1978-31229 DON JUAN CARLOS I, REY DE ESPAÑA, A TODOS LOS QUE LA PRESENTE VIEREN Y ENTENDIEREN, SABED: QUE LAS CORTES HAN APROBADO Y EL PUEBLO ESPAÑOL RATIFICADO LA SIGUIENTE CONSTITUCIÓN: PREÁMBULO La Nación española, deseando establecer la justicia, la libertad y la seguridad y promover el bien de cuantos la integran, en uso de su soberanía, proclama su voluntad de: Garantizar la convivencia democrática dentro de la Constitución y de las leyes conforme a un orden económico y social justo.Consolidar un Estado de Derecho que asegure el imperio de la ley como expresión de la voluntad popular.Proteger a todos los españoles y pueblos de España en el ejercicio de los derechos humanos, sus culturas y tradiciones, lenguas e instituciones.Promover el progreso de la cultura y de la economía para asegurar a tod

In [ ]:
random.sample(pages_and_chunks_over_min_token_len, k=1)

[{'page_number': 21,
  'sentence_chunk': 'Las sentencias serán siempre motivadas y se pronunciarán en audiencia pública.CONSTITUCIÓN ESPAÑOLA § 1 Constitución Española – 22 –',
  'chunk_char_count': 132,
  'chunk_word_count': 19,
  'chunk_token_count': 33.0}]

### Embediendo nuestros chunks de texto

Los embeddings de texto significan que textos con significado similar tienen representación numérica similar.


Nuestro objetivo es convertir cada uno de nuestros chunks en una representación numérica (un vector de embedding, donde un vector es una secuencia de números organizados en orden).

Usaremos nuestras computadoras para encontrar patrones en los embeddings y luego podemos usar sus mapeos de texto para profundizar nuestro entendimiento.

Usaremos la librería [`sentence-transformers`](https://www.sbert.net/docs/installation.html) que contiene muchos modelos de embedding pre-entrenados.

Específicamente, obtendremos el modelo `all-mpnet-base-v2` (puedes ver el uso previsto del modelo en la [tarjeta del modelo de Hugging Face](https://huggingface.co/sentence-transformers/all-mpnet-base-v2#intended-uses)).

In [ ]:
!pip install sentence-transformers # for embedding models

In [ ]:
from sentence_transformers import SentenceTransformer

try:
  embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device="cuda")
except:
  embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2")

for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
100%|██████████| 128/128 [00:03<00:00, 33.72it/s]


Nuestro embedding tiene una forma de `(768,)` lo que significa que es un vector de 768 números que representan nuestro texto en un espacio de alta dimensión.

In [ ]:
  pages_and_chunks_over_min_token_len[0]["embedding"].shape

(768,)

Viendo un chunk por ejemplo el número 9, vemos que tiene suficiente longitud como para expresar una idea por completo, pero tampoco es tan pequeño como para que no tenga sentido. Recordemos que queremos disminuir la longitud de las sentences para prosesarlas mejor, pero no tanto que rompamos la idea.

In [ ]:
# Convertimos el texto de los chunks en un lista simple
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]
text_chunks[9]

'Nadie puede ser privado de su libertad, sino con la observancia de lo establecido en este artículo y en los casos y en la forma previstos en la ley.2.La detención preventiva no podrá durar más del tiempo estrictamente necesario para la realización de las averiguaciones tendentes al esclarecimiento de los hechos, y, en todo caso, en el plazo máximo de setenta y dos horas, el detenido deberá ser puesto en libertad o a disposición de la autoridad judicial.3.Toda persona detenida debe ser informada de forma inmediata, y de modo que le sea comprensible, de sus derechos y de las razones de su detención, no pudiendo ser obligada a declarar.Se garantiza la asistencia de abogado al detenido en las diligencias policiales y judiciales, en los términos que la ley establezca.4.La ley regulará un procedimiento de «habeas corpus» para producir la inmediata puesta a disposición judicial de toda persona detenida ilegalmente.Asimismo, por ley se determinará el plazo máximo de duración de la prisión pro

In [ ]:
len(text_chunks)

128

Aquí vamos a revisar cómo esta "codificado o numerado" el embeddings (vectores densos) número 0 del diccionario.

In [ ]:
text_chunk_embeddings = embedding_model.encode(text_chunks,
                                               batch_size=16, # Embed all texts in batches
                                               convert_to_tensor=True)
text_chunk_embeddings[0]

tensor([-3.1860e-02, -1.8373e-02,  2.7644e-02, -2.1847e-02, -7.2178e-03,
         3.4185e-02, -2.0138e-02,  4.8112e-03,  3.4348e-02,  1.5689e-02,
        -3.7223e-02,  4.0855e-02,  1.7537e-02, -5.5459e-02, -1.7907e-02,
         2.4197e-02,  1.0484e-03,  7.1884e-02,  6.8235e-03, -1.8812e-03,
        -9.6524e-02,  7.7338e-03,  3.2286e-02, -7.6648e-04,  6.9055e-04,
         5.7173e-02,  2.9790e-02, -6.7637e-02, -3.4502e-02,  1.4148e-02,
         4.9238e-02,  3.0116e-02, -6.5304e-03, -3.8688e-02,  2.1239e-06,
        -1.0710e-02, -1.8679e-02,  1.0649e-02,  6.2567e-02, -3.7064e-02,
         1.9015e-02, -2.4790e-02,  6.6606e-03,  1.4286e-02, -8.0166e-02,
        -3.9964e-02,  7.3184e-03, -1.0953e-02,  7.1045e-02, -6.3321e-02,
        -1.4946e-02, -1.1966e-02,  9.0167e-03, -5.4875e-02, -2.1327e-02,
         3.1292e-02,  3.4566e-02, -4.5531e-03,  1.1959e-02,  6.1582e-02,
         3.9975e-02,  9.0415e-04,  2.7609e-02, -3.8020e-02, -1.0030e-01,
        -8.0439e-02, -3.4899e-02, -3.9426e-02,  7.0

Aquí vamos a guardar todos esos embedding que hemos creado en un fichero .csv

In [ ]:
#Saving embedding to file
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(save_path, index=False)

Luego los recuperamos de nuevo y los desplegamos el texto que tenemos como una doble comprobación.

In [ ]:
# Import saved file and view
text_chunks_and_embeddings_df_load = pd.read_csv(save_path)
text_chunks_and_embeddings_df_load.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,0,§ 1 Constitución Española Cortes Generales «BO...,1571,246,392.75,[-3.18596661e-02 -1.83727704e-02 2.76439786e-...
1,1,Artículo 2. La Constitución se fundamenta en ...,695,108,173.75,[-2.28652544e-02 1.24958679e-02 5.83437784e-...
2,1,La bandera de España está formada por tres fra...,756,118,189.00,[-2.86408141e-02 -1.46402894e-02 -2.01208349e-...
3,1,Artículo 7. Los sindicatos de trabajadores y ...,828,125,207.00,[-3.80014069e-02 2.02068556e-02 3.87908071e-...
4,1,2.Corresponde a los poderes públicos promover ...,708,104,177.00,[-2.54121330e-02 4.12110947e-02 1.10657094e-...


# RAG - Búsqueda y Respuesta

### Búsqueda por similitud
La búsqueda por similitud o búsqueda semántica o búsqueda vectorial es la idea de buscar por *semántica*.

Con la búsqueda por palabras clave, estás tratando de hacer coincidir la cadena "manzana" con la cadena "manzana".

Mientras que con la búsqueda por similitud/semántica, podrías querer buscar "funciones del presidente".
Y obtener resultados que no necesariamente contienen las palabras "funciones del presidente" pero obtener fragmentos de texto que coinciden con ese significado.


In [ ]:
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

text_chunks_and_embedding_df = pd.read_csv(save_path)
#convert embedding to array (it got converted to string when it saved)
text_chunks_and_embedding_df["embedding"] = text_chunks_and_embedding_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

#converting embedding into torch tensor
embeddings = torch.tensor(np.stack(text_chunks_and_embedding_df["embedding"].tolist(), axis=0), dtype=torch.float32).to(device)
# Convert texts and embedding df to list of dicts
pages_and_chunks = text = text_chunks_and_embedding_df.to_dict(orient="records")

text_chunks_and_embeddings_df

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,0,§ 1 Constitución Española Cortes Generales «BO...,1571,246,392.75,"[-0.031859666, -0.01837277, 0.027643979, -0.02..."
1,1,Artículo 2. La Constitución se fundamenta en ...,695,108,173.75,"[-0.022865254, 0.012495868, 0.005834378, 0.010..."
2,1,La bandera de España está formada por tres fra...,756,118,189.00,"[-0.028640814, -0.014640289, -0.020120835, 0.0..."
3,1,Artículo 7. Los sindicatos de trabajadores y ...,828,125,207.00,"[-0.038001407, 0.020206856, 0.038790807, 0.019..."
4,1,2.Corresponde a los poderes públicos promover ...,708,104,177.00,"[-0.025412133, 0.041211095, 0.011065709, -0.01..."
...,...,...,...,...,...,...
123,35,"prevé en el apartado 2 del artículo 148, cuand...",1373,211,343.25,"[-0.00536294, 0.033508472, 0.019370528, -0.024..."
124,35,Las ciudades de Ceuta y Melilla podrán constit...,1417,226,354.25,"[-0.011080715, 0.016789012, -0.009523982, -0.0..."
125,35,A los efectos de lo establecido en el artículo...,355,58,88.75,"[-0.031591877, 0.03561912, 0.01018097, -0.0552..."
126,36,"Durante este período, el actual Presidente del...",2225,390,556.25,"[-0.012609532, 0.020806761, 0.0041038482, -0.0..."


In [ ]:
embeddings.shape

torch.Size([128, 768])

La recuperación se realiza siguiendo estos pasos:
1. Definir una cadena de consulta (query).
2. Convertir la cadena de consulta en un embedding con el mismo modelo que usamos para embeber nuestros chunks de texto.
3. Realizar una función de [producto punto](https://pytorch.org/docs/stable/generated/torch.dot.html) o [similitud de coseno](https://en.wikipedia.org/wiki/Cosine_similarity) entre los embeddings de texto y el embedding de la consulta para obtener puntuaciones de similitud.
4. Ordenar los resultados del paso 3 en orden descendente (una puntuación más alta significa más similitud a los ojos del modelo) y usar estos valores para inspeccionar los textos.

In [ ]:
from sentence_transformers import util

query = "nacionalidad española"
print(f"Query : {query}")

# Use the 'device' variable to ensure the query embedding is on the same device as the text embeddings
query_embedding = embedding_model.encode(query, convert_to_tensor=True).to(device)

dot_scores = util.dot_score(query_embedding, embeddings)[0]

top_results = torch.topk(dot_scores, k=5)
top_results

Query : nacionalidad española


torch.return_types.topk(
values=tensor([0.6881, 0.6621, 0.6434, 0.6394, 0.6378], device='cuda:0'),
indices=tensor([ 5,  4,  1, 31,  6], device='cuda:0'))

In [ ]:
for score, idx in zip(top_results[0], top_results[1]):
    print(f"Score: {score:.4f}")
    print("Text")
    print(pages_and_chunks[idx]["sentence_chunk"])
    print("\n\n")


Score: 0.6881
Text
TÍTULO I De los derechos y deberes fundamentales Artículo 10.  1.La dignidad de la persona, los derechos inviolables que le son inherentes, el libre desarrollo de la personalidad, el respeto a la ley y a los derechos de los demás son fundamento del orden político y de la paz social.2.Las normas relativas a los derechos fundamentales y a las libertades que la Constitución reconoce se interpretarán de conformidad con la Declaración Universal de Derechos Humanos y los tratados y acuerdos internacionales sobre las mismas materias ratificados por España.CAPÍTULO PRIMERO De los españoles y los extranjeros Artículo 11.  1.La nacionalidad española se adquiere, se conserva y se pierde de acuerdo con lo establecido por la ley.2.Ningún español de origen podrá ser privado de su nacionalidad.3.El Estado podrá concertar tratados de doble nacionalidad con los países iberoamericanos o con aquellos que hayan tenido o tengan una particular vinculación con España.



Score: 0.6621
Text

In [ ]:
from sentence_transformers import util

def retrieve_relevant_resources(query: str, n_resources_to_return: int=5):
    """
    Embeds a query with model and returns top k scores and indices from embeddings.
    """
    # Use the 'device' variable to ensure the query embedding is on the same device as the text embeddings
    query_embedding = embedding_model.encode(query, convert_to_tensor=True).to(device)

    dot_scores = util.dot_score(query_embedding, embeddings)[0]

    scores, indices = torch.topk(dot_scores, k=n_resources_to_return)

    return scores, indices

In [ ]:
retrieve_relevant_resources(query)

(tensor([0.6881, 0.6621, 0.6434, 0.6394, 0.6378], device='cuda:0'),
 tensor([ 5,  4,  1, 31,  6], device='cuda:0'))

In [ ]:
def print_top_results_and_scores(query: str, n_resources_to_return: int=5):
    """
    Takes a query, retrieves most relevant resources and prints them out in descending order.
    """
    scores, indices = retrieve_relevant_resources(query, n_resources_to_return=n_resources_to_return)
    for score, idx in zip(scores, indices):
        print(f"Score: {score:.4f}")
        print("Text")
        print(pages_and_chunks[idx]["sentence_chunk"])
        print("\n\n")

In [ ]:
print_top_results_and_scores(query)

Score: 0.6881
Text
TÍTULO I De los derechos y deberes fundamentales Artículo 10.  1.La dignidad de la persona, los derechos inviolables que le son inherentes, el libre desarrollo de la personalidad, el respeto a la ley y a los derechos de los demás son fundamento del orden político y de la paz social.2.Las normas relativas a los derechos fundamentales y a las libertades que la Constitución reconoce se interpretarán de conformidad con la Declaración Universal de Derechos Humanos y los tratados y acuerdos internacionales sobre las mismas materias ratificados por España.CAPÍTULO PRIMERO De los españoles y los extranjeros Artículo 11.  1.La nacionalidad española se adquiere, se conserva y se pierde de acuerdo con lo establecido por la ley.2.Ningún español de origen podrá ser privado de su nacionalidad.3.El Estado podrá concertar tratados de doble nacionalidad con los países iberoamericanos o con aquellos que hayan tenido o tengan una particular vinculación con España.



Score: 0.6621
Text

# Instalando Gemma-2b
Usaremos Gemma_instruct_2b para esto.

In [ ]:
!pip install -q -U keras-nlp
!pip install -q -U keras>=3

import os

os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

In [ ]:
import keras
import keras_nlp
import kagglehub
import os

# Access your Kaggle API credentials from Colab secrets
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

# Print environment variables to check if they are set
print(f"KAGGLE_USERNAME set: {os.environ.get('KAGGLE_USERNAME') is not None}")
print(f"KAGGLE_KEY set: {os.environ.get('KAGGLE_KEY') is not None}")


# Download the model using kagglehub
# Replace 'keras/gemma/keras/gemma_instruct_2b_en/3' with the correct model handle if needed
model_path = kagglehub.model_download('keras/gemma/keras/gemma_instruct_2b_en/3')
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset(model_path)

KAGGLE_USERNAME set: True
KAGGLE_KEY set: True


**Aquí es donde podemos hacer nuestra pregunta al modelo, lo haremos contestando Sin RAG y Con RAG**

In [ ]:
# NOTA: Esta respuesta es SIN contexto de la Constitución
# El modelo responde basándose solo en su conocimiento general
input_text = "¿es posible tener la nacionalidad española junto a otra segunda nacionalidad?"

outputs = gemma_lm.generate(input_text, max_length=256)
print(" Respuesta SIN RAG (solo conocimiento del modelo):")
print(outputs)
print(" Esta respuesta puede no ser precisa o estar desactualizada.")
print("Compárala con la respuesta CON RAG más adelante.")

 Respuesta SIN RAG (solo conocimiento del modelo):
¿es posible tener la nacionalidad española junto a otra segunda nacionalidad?

Sí, es posible tener la nacionalidad española junto a otra segunda nacionalidad. La nacionalidad española es una nacionalidad legal que se concede por nacimiento en España o por legitimación. Una persona puede ser nacionalizada española si cumple con los requisitos legales establecidos por el Estado.
 Esta respuesta puede no ser precisa o estar desactualizada.
Compárala con la respuesta CON RAG más adelante.


**Aquí vamos a crear una serie de preguntas sobre la Constitución Española que nos ayuden a probar el RAG**

### Aumentando nuestro prompt con diferentes contextos

  Nos gustaría que el aumento de la respuesta tomara los resultados de nuestra busqueda y los más relevante de la busqueda y lo pusiera dentro del prompt que vamos a pasarle a nuestro LLM.


#### Queremos un prompt que tenga la forma de lista y le damos un ejemplo

Basado en el siguiente contexto:
- item1
- item2
- item3
- item4
- item... etc. etc.

Por favor contesa a la siguiente pregunta: ¿cuales son los principales artículos de la contstitución?

Respuesta:

In [ ]:
def prompt_formatter(query: str,
                     context_items: list[dict]) -> str:
    """
    Aumenta la query con contexto textual de context_items.
    """
    # Unir los items de contexto en un párrafo con viñetas
    context = "- " + "\n- ".join([item["sentence_chunk"] for item in context_items])

    # Crear un prompt base con ejemplos para ayudar al modelo
    base_prompt = """Basándote en los siguientes fragmentos de contexto, por favor responde a la pregunta.
Tómate tiempo para pensar extrayendo los pasajes relevantes del contexto antes de responder.
No devuelvas el razonamiento, solo devuelve la respuesta final.
Asegúrate de que tus respuestas sean lo más explicativas posible.
Usa los siguientes ejemplos como referencia para el estilo de respuesta ideal.

Ejemplo 1:
Pregunta: ¿Cuáles son los derechos fundamentales según la Constitución?
Respuesta: Los derechos fundamentales recogidos en la Constitución Española incluyen el derecho a la vida y a la integridad física y moral, la libertad ideológica, religiosa y de culto, el derecho a la libertad y a la seguridad, el derecho al honor, a la intimidad personal y familiar y a la propia imagen, la inviolabilidad del domicilio, el secreto de las comunicaciones, la libertad de residencia y circulación, la libertad de expresión, el derecho de reunión, el derecho de asociación, el derecho a la participación política, el derecho a la tutela judicial efectiva, y la prohibición de discriminación por razón de nacimiento, raza, sexo, religión, opinión o cualquier otra condición o circunstancia personal o social.

Ejemplo 2:
Pregunta: ¿Cómo se reforma la Constitución Española?
Respuesta: La reforma constitucional puede iniciarse por el Gobierno, el Congreso de los Diputados, el Senado o las Asambleas de las Comunidades Autónomas. La reforma debe ser aprobada por mayoría de tres quintos de cada una de las Cámaras. Si no hay acuerdo, se intentará mediante comisión mixta. En caso de reforma total o parcial que afecte al Título Preliminar, Capítulo II Sección 1ª del Título I o al Título II, se requerirá aprobación por dos tercios de cada Cámara, disolución de las Cortes, ratificación por las nuevas Cámaras y referéndum obligatorio.

Ejemplo 3:
Pregunta: ¿Qué funciones tiene el Tribunal Constitucional?
Respuesta: El Tribunal Constitucional es el intérprete supremo de la Constitución. Sus funciones principales incluyen: conocer del recurso de inconstitucionalidad contra leyes y disposiciones normativas con fuerza de ley, conocer del recurso de amparo por violación de derechos y libertades fundamentales, resolver conflictos de competencia entre el Estado y las Comunidades Autónomas o de los de éstas entre sí, y cualquier otra materia que le atribuyan la Constitución o las leyes orgánicas.

Ahora usa los siguientes fragmentos de contexto para responder a la pregunta del usuario:
{context}

Pasajes relevantes: <extrae los pasajes relevantes del contexto aquí>
Pregunta del usuario: {query}
Respuesta:"""

    # Actualizar el prompt base con los items de contexto y la query
    prompt = base_prompt.format(context=context, query=query)

    return prompt

In [ ]:
# Preguntas para probar el sistema RAG - Constitución Española
query_list = [
    "¿Es posible tener la nacionalidad española junto a otra segunda nacionalidad?",
    "¿Cuáles son los derechos fundamentales de los ciudadanos españoles?",
    "¿Cómo se organiza territorialmente España según la Constitución?",
    "¿Qué funciones tiene el Rey en la Constitución Española?",
    "¿Cómo se aprueba una ley orgánica?",
    "¿Qué es el Tribunal Constitucional y cuáles son sus funciones principales?",
    "¿Puede el Congreso destituir al Presidente del Gobierno?",
    "¿Qué edad mínima se requiere para ser Diputado?",
    "¿Cómo se eligen los Senadores en España?",
    "¿Qué derechos tienen los extranjeros en España según la Constitución?"
]

In [ ]:
# Seleccionar una pregunta aleatoria de la lista de preguntas de la Constitución
query = random.choice(query_list)
print(f"Query: {query}")

# Obtener recursos relevantes (fragmentos de texto de la Constitución)
# retrieve_relevant_resources ya usa los embeddings de la Constitución
scores, indices = retrieve_relevant_resources(query=query)

# Crear una lista de context_items usando los fragmentos de la Constitución
context_items = [pages_and_chunks[i] for i in indices]

# Formatear el prompt con la pregunta y los fragmentos de la Constitución
prompt = prompt_formatter(query=query,
                          context_items=context_items)
print(prompt)

Query: ¿Cómo se organiza territorialmente España según la Constitución?
Basándote en los siguientes fragmentos de contexto, por favor responde a la pregunta.
Tómate tiempo para pensar extrayendo los pasajes relevantes del contexto antes de responder.
No devuelvas el razonamiento, solo devuelve la respuesta final.
Asegúrate de que tus respuestas sean lo más explicativas posible.
Usa los siguientes ejemplos como referencia para el estilo de respuesta ideal.

Ejemplo 1:
Pregunta: ¿Cuáles son los derechos fundamentales según la Constitución?
Respuesta: Los derechos fundamentales recogidos en la Constitución Española incluyen el derecho a la vida y a la integridad física y moral, la libertad ideológica, religiosa y de culto, el derecho a la libertad y a la seguridad, el derecho al honor, a la intimidad personal y familiar y a la propia imagen, la inviolabilidad del domicilio, el secreto de las comunicaciones, la libertad de residencia y circulación, la libertad de expresión, el derecho de r

In [ ]:
outputs = gemma_lm.generate(prompt,max_length=2048)

print(f"Query: {query}")
print(f"RAG answer:\n{outputs.replace(prompt, '')}")

Query: ¿Cómo se organiza territorialmente España según la Constitución?
RAG answer:
 No hay información en el contexto sobre cómo se organiza territorialmente España según la Constitución.


In [ ]:
def ask(query, return_answer_only=True):
    """
    Toma una query, encuentra recursos/contexto relevantes y genera una respuesta basada en ellos.
    """
    # Obtener scores e indices de resultados más relacionados
    scores, indices = retrieve_relevant_resources(query=query)

    # Crear lista de context items
    context_items = [pages_and_chunks[i] for i in indices]

    # Añadir score a cada context item
    for i, item in enumerate(context_items):
        item["score"] = scores[i].cpu()  # Retornar score a CPU

    # Formatear el prompt con los context items
    prompt = prompt_formatter(query=query,
                              context_items=context_items)

    # Generar respuesta
    outputs = gemma_lm.generate(prompt, max_length=2048)

    # Eliminar el prompt del output
    output_text = outputs.replace(prompt, "")

    # Extraer páginas de origen (metadata)
    source_pages = list(set([item['page_number'] for item in context_items]))
    source_pages.sort()

    # Agregar información de fuentes al final de la respuesta
    sources_info = f" Fuentes consultadas: Páginas {', '.join(map(str, source_pages))} de la Constitución Española"
    output_with_sources = output_text + sources_info

    # Retornar solo la respuesta o con context items
    if return_answer_only:
        return output_with_sources

    return output_with_sources, context_items

In [ ]:
query = random.choice(query_list)
print(f"Query: {query}")

return_answer_only = True
if return_answer_only:
    # Answer query with context and return context
    answer = ask(query=query, return_answer_only=True)
    print(f"Answer:\n")
    print(answer)
else:
    answer, context_items = ask(query=query, return_answer_only=False)
    print(f"Answer:\n")
    print(answer)
    print(f"\n\nContext items:")
    print(context_items)

Query: ¿Es posible tener la nacionalidad española junto a otra segunda nacionalidad?
Answer:

 No, la nacionalidad española solo se adquiere, conserva y pierde de acuerdo con lo establecido por la ley. Fuentes consultadas: Páginas 1, 2, 18, 25 de la Constitución Española


# Conclusión:

In [ ]:
query = "¿Es posible tener la nacionalidad española junto a otra segunda nacionalidad?"
print(f"Query: {query}")

return_answer_only = True
if return_answer_only:
    # Answer query with context and return context
    answer = ask(query=query, return_answer_only=True)
    print(f"Answer:\n")
    print(answer)
else:
    answer, context_items = ask(query=query, return_answer_only=False)
    print(f"Answer:\n")
    print(answer)
    print(f"\n\nContext items:")
    print(context_items)

Query: ¿Es posible tener la nacionalidad española junto a otra segunda nacionalidad?
Answer:

 No, la nacionalidad española solo se adquiere, conserva y pierde de acuerdo con lo establecido por la ley. Fuentes consultadas: Páginas 1, 2, 18, 25 de la Constitución Española


**En resumen este es un pequeña guia sobre los pasos para realizar un RAG que sirva para hacer Question & Answer, existen otros tipos de RAG que sirve para realizar resúmenes, o para alamcenar incluso información multimodal, texto e imágenes.**

EOF